# Entanglement: correlations no coin can fake

**Professor Nano Labs · Quantum computing**

**Try it live in your browser:** [Entanglement lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/entanglement-bell-test/)
**Watch first:** [Quantum entanglement; exposing its key features](https://www.youtube.com/watch?v=9vDyS2_Hufk)
**Watch later:** [Two qubit state: Separable vs Entangled vs Bell states](https://www.youtube.com/watch?v=4BRQsrh4oGw) · [Two and three qubits quantum gates](https://www.youtube.com/watch?v=iMjpZwISIlA) · [Quick introduction to the density matrix in quantum mechanics](https://www.youtube.com/watch?v=ma136N8d3Xc)

A source sends one spin to Alice and one to Bob, in the singlet state
$$|\Psi^-\rangle=\frac{|{\uparrow}{\downarrow}\rangle-|{\downarrow}{\uparrow}\rangle}{\sqrt2}.$$
Each measures spin along a dial, $\hat n(\theta)=(\sin\theta,0,\cos\theta)$, and gets +1 (up along the dial) or −1. Quantum mechanics predicts that each alone sees a fair coin, while together $E(a,b)=\langle AB\rangle=-\cos(a-b)$.

We compare the pair with two classical stand-ins: **socks** (pairs that are simply opposite along z) and the best **answer sheet** (Bell's hidden-arrow model, where each pair carries instructions fixed at the source). The four sections match the lab's four tabs. It needs only NumPy and Matplotlib. **Run all cells from the top.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]])
Z = np.diag([1.0, -1.0]).astype(complex)
I2 = np.eye(2)
up, dn = np.array([1, 0], dtype=complex), np.array([0, 1], dtype=complex)
singlet = (np.kron(up, dn) - np.kron(dn, up)) / np.sqrt(2)

def dial(theta, s):
    # projector onto spin s (+1 or -1) along n = (sin theta, 0, cos theta)
    return 0.5 * (I2 + s * (np.sin(theta) * X + np.cos(theta) * Z))

def rho_pair(v=1.0):
    # the singlet mixed with white noise, purity v
    return v * np.outer(singlet, singlet.conj()) + (1 - v) * np.eye(4) / 4

rho_socks = 0.5 * (np.outer(np.kron(up, dn), np.kron(up, dn)) + np.outer(np.kron(dn, up), np.kron(dn, up)))

def joint(rho, a, b):
    # [P(++), P(+-), P(-+), P(--)], Alice first
    return np.real([np.trace(rho @ np.kron(dial(a, s), dial(b, t))) for s in (1, -1) for t in (1, -1)])

def corr(rho, a, b):
    p = joint(rho, a, b)
    return p[0] - p[1] - p[2] + p[3]

def corr_arrow(a, b, n=200000):
    # Bell's hidden arrow: Alice answers sign cos(a - lam), Bob -sign cos(b - lam); lam uniform
    lam = (np.arange(n) + 0.5) * 2 * np.pi / n
    A = np.where(np.cos(a - lam) >= 0, 1, -1); B = -np.where(np.cos(b - lam) >= 0, 1, -1)
    return np.mean(A * B)

deg = np.pi / 180
print('singlet, dials 0 and 60 degrees:', joint(rho_pair(), 0, 60 * deg), ' E =', round(corr(rho_pair(), 0, 60 * deg), 4))
print("Alice's own chance of +1:", round(sum(joint(rho_pair(), 0.3, 1.7)[:2]), 6))

## 1. Same dial, any direction

With both dials at the same angle, the singlet is perfectly anticorrelated along *every* axis, $E=-1$. The socks are anticorrelated only along z, and $E_{\rm socks}(\theta,\theta)=-\cos^2\theta$ falls to zero along x. That is the difference between classical correlation and entanglement.

In [ ]:
th = np.linspace(0, np.pi, 181)
plt.figure(figsize=(7, 3.6))
plt.plot(th / deg, [corr(rho_pair(), t, t) for t in th], color='#a3326c', lw=2.4, label='entangled pair')
plt.plot(th / deg, [corr(rho_socks, t, t) for t in th], '--', color='#c47a12', lw=2, label='socks')
plt.xlabel('Both dials at θ from z (°)'); plt.ylabel('Correlation E'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()
for t in (0, 45, 90):
    print(f'theta = {t:2d} deg: pair E = {corr(rho_pair(), t * deg, t * deg):+.3f}, socks E = {corr(rho_socks, t * deg, t * deg):+.3f}')

## 2. Turn one dial

As Bob's dial turns away from Alice's, the singlet's correlation follows $-\cos\theta$. Bell's hidden-arrow model, the best answer sheet that still gives perfect anticorrelation at equal dials, gives the straight line $-(1-2\theta/\pi)$. They meet at 0°, 90° and 180° and differ in between.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
th = np.linspace(0, np.pi, 91)
Eq = np.array([corr(rho_pair(), 0, t) for t in th])
Ea = np.array([corr_arrow(0, t) for t in th])
a1.plot(th / deg, Eq, color='#a3326c', lw=2.4, label='entangled pair: −cos θ')
a1.plot(th / deg, Ea, '--', color='#c47a12', lw=2, label='answer sheets: −(1 − 2θ/π)')
a1.set_xlabel('Angle between the dials θ (°)'); a1.set_ylabel('Correlation E'); a1.legend(); a1.grid(alpha=0.3)
a2.plot(th / deg, (1 + Eq) / 2, color='#a3326c', lw=2.4, label='entangled pair: sin²(θ/2)')
a2.plot(th / deg, (1 + Ea) / 2, '--', color='#c47a12', lw=2, label='answer sheets: θ/π')
a2.set_xlabel('Angle between the dials θ (°)'); a2.set_ylabel('P(same answer)'); a2.legend(); a2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

gap = Eq - Ea
i = np.argmin(gap)
print('largest gap (quantum - sheets) =', round(gap[i], 3), 'at', round(th[i] / deg, 1), 'deg; exact at asin(2/pi) =', round(np.degrees(np.arcsin(2 / np.pi)), 2), 'deg')
print('at 60 deg: P(same) pair', round((1 + corr(rho_pair(), 0, 60 * deg)) / 2, 4), ' sheets', round((1 + corr_arrow(0, 60 * deg)) / 2, 4))

## 3. The CHSH game

Each round Alice uses $a$ or $a'$ and Bob $b$ or $b'$, at random, and the score is
$$S=E(a,b)-E(a,b')+E(a',b)+E(a',b').$$
An answer sheet fixes four ±1 answers in advance, so every sheet scores exactly ±2, and any mix of sheets scores at most 2. The singlet reaches $2\sqrt2$ at $a=0$, $a'=90°$, $b=45°$, $b'=135°$. Mixed with noise to purity $v$ it scores $2\sqrt2\,v$, so the violation vanishes at $v=1/\sqrt2$, although the state stays entangled down to $v=1/3$ (where its partial transpose stops having a negative eigenvalue).

In [ ]:
sheets = [(Aa, Aap, Bb, Bbp, Aa * Bb - Aa * Bbp + Aap * Bb + Aap * Bbp)
          for Aa in (1, -1) for Aap in (1, -1) for Bb in (1, -1) for Bbp in (1, -1)]
print('scores of all 16 answer sheets:', sorted(set(s[-1] for s in sheets)))

def chsh(rho, a, ap, b, bp):
    return corr(rho, a, b) - corr(rho, a, bp) + corr(rho, ap, b) + corr(rho, ap, bp)
best = (0, 90 * deg, 45 * deg, 135 * deg)
print('singlet at the best dials: |S| =', round(abs(chsh(rho_pair(), *best)), 4), ' 2*sqrt(2) =', round(2 * np.sqrt(2), 4))

# simulated rounds
rng = np.random.default_rng(11)
def play(rho, dials, rounds):
    a, ap, b, bp = dials
    sums = {k: [0, 0] for k in ('ab', 'abp', 'apb', 'apbp')}
    for _ in range(rounds):
        ia, ib = rng.integers(2), rng.integers(2)
        A_ang, B_ang = (a, ap)[ia], (b, bp)[ib]
        p = joint(rho, A_ang, B_ang)
        k = rng.choice(4, p=p)
        AB = 1 if k in (0, 3) else -1
        key = ('a' if ia == 0 else 'ap') + ('b' if ib == 0 else 'bp')
        sums[key][0] += 1; sums[key][1] += AB
    E = {k: s / n for k, (n, s) in sums.items()}
    err = {k: np.sqrt((1 - E[k]**2) / sums[k][0]) for k in sums}
    S = E['ab'] - E['abp'] + E['apb'] + E['apbp']
    return abs(S), np.sqrt(sum(e**2 for e in err.values()))
S, dS = play(rho_pair(), best, 10000)
print(f'10,000 simulated rounds: |S| = {S:.3f} +- {dS:.3f}, {(S - 2) / dS:.0f} standard deviations above 2')

# score against the step between evenly spaced dials, and against purity
phi = np.linspace(0, np.pi / 2, 91)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(phi / deg, [abs(chsh(rho_pair(), 0, 2 * f, f, 3 * f)) for f in phi], color='#a3326c', lw=2.4, label='entangled pair')
a1.plot(phi / deg, [abs(corr_arrow(0, f, 40000) - corr_arrow(0, 3 * f, 40000) + corr_arrow(2 * f, f, 40000) + corr_arrow(2 * f, 3 * f, 40000)) for f in phi], '--', color='#c47a12', lw=2, label='answer sheets')
a1.axhline(2, color='#c47a12', lw=1); a1.set_xlabel('Step φ: a = 0, b = φ, a′ = 2φ, b′ = 3φ (°)'); a1.set_ylabel('|S|'); a1.legend(); a1.grid(alpha=0.3)
v = np.linspace(0, 1, 101)
def min_eig_partial_transpose(r):
    rt = r.reshape(2, 2, 2, 2).transpose(0, 3, 2, 1).reshape(4, 4)     # transpose Bob's spin
    return np.linalg.eigvalsh(rt).min()
a2.plot(v, [abs(chsh(rho_pair(x), *best)) for x in v], color='#a3326c', lw=2.4, label='best |S| = 2√2 v')
a2.axhline(2, color='#c47a12', lw=1.2, label='classical limit')
a2.axvspan(1 / 3, 1, color='#a3326c', alpha=0.07, label='entangled (v > 1/3)')
a2.set_xlabel('Purity v'); a2.set_ylabel('|S|'); a2.legend(); a2.grid(alpha=0.3)
plt.tight_layout(); plt.show()
for x in (0.3, 0.34, 0.6, 0.71, 0.72):
    print(f'v = {x:.2f}: |S| = {abs(chsh(rho_pair(x), *best)):.3f}, smallest eigenvalue of the partial transpose = {min_eig_partial_transpose(rho_pair(x)):+.4f}')

## 4. Steering, not signalling

When Alice gets $A$ along $a$, Bob's spin is left in the state found by projecting Alice's spin and tracing it out: its Bloch vector is $-A\,v\,\hat n(a)$. Averaged over Alice's two equally likely results it is zero, the maximally mixed state $I/2$, whatever Alice's dial. So Bob's own statistics never depend on Alice's choice.

In [ ]:
def bob_given(rho, a, A):
    M = np.kron(dial(a, A), I2) @ rho @ np.kron(dial(a, A), I2)
    pA = np.trace(M).real
    rb = np.trace((M / pA).reshape(2, 2, 2, 2), axis1=0, axis2=2)
    return pA, np.real([np.trace(rb @ s) for s in (X, Y, Z)])

for a in (0, 90 * deg, 37 * deg):
    pp, rp = bob_given(rho_pair(), a, 1)
    pm, rm = bob_given(rho_pair(), a, -1)
    avg = pp * rp + pm * rm
    print(f'Alice at {a / deg:4.0f} deg: Bob given +1 -> {rp}, given -1 -> {rm}, average {avg}')

# Bob's chance of +1 along b = 90 deg, whatever Alice does
b = 90 * deg
for a in (0, 45 * deg, 90 * deg):
    p = joint(rho_pair(), a, b)
    print(f'Alice at {a / deg:3.0f} deg: Bob P(+1) = {p[0] + p[2]:.3f}; given Alice +1: {abs(p[0] / (p[0] + p[1])):.3f}; given Alice -1: {abs(p[2] / (p[2] + p[3])):.3f}')

## 5. Checks

The numbers quoted in the lab's "Predict first" answers, recomputed here.

In [ ]:
# Q1: along x the pair is still perfectly opposite; socks are uncorrelated
assert np.isclose(corr(rho_pair(), 90 * deg, 90 * deg), -1) and np.isclose(corr(rho_socks, 90 * deg, 90 * deg), 0)
# Q2: Alice alone always sees a fair coin
assert all(np.isclose(sum(joint(r, a, b)[:2]), 0.5) for r in (rho_pair(), rho_socks) for a, b in ((0, 0), (1, 2)))
# Q3: 60 degrees: 25% vs 1/3; about four standard deviations apart with 1000 pairs each
pq, pc = (1 + corr(rho_pair(), 0, 60 * deg)) / 2, (1 + corr_arrow(0, 60 * deg)) / 2
assert np.isclose(pq, 0.25) and abs(pc - 1 / 3) < 1e-4
print('Q3 separation:', round((pc - pq) / np.sqrt(pq * (1 - pq) / 1000 + pc * (1 - pc) / 1000), 1), 'standard deviations')
# Q4: largest gap about 0.21 near 40 degrees
assert abs(gap.min() + 0.21) < 0.005 and abs(np.degrees(np.arcsin(2 / np.pi)) - 39.5) < 0.1
# Q5: every sheet scores +-2
assert sorted(set(s[-1] for s in sheets)) == [-2, 2]
# Q6: 2 sqrt 2, and about +-0.03 after 10,000 rounds
assert np.isclose(abs(chsh(rho_pair(), *best)), 2 * np.sqrt(2)) and abs(dS - 0.028) < 0.003
# Q7: 2 sqrt 2 v; at v = 0.6 about 1.70; entangled down to 1/3
assert np.isclose(abs(chsh(rho_pair(0.6), *best)), 2 * np.sqrt(2) * 0.6)
assert min_eig_partial_transpose(rho_pair(0.34)) < 0 < min_eig_partial_transpose(rho_pair(0.33))
# Q8: Bob's own chance of +1 is 1/2 for any dial Alice picks
assert all(np.isclose(joint(rho_pair(), a, 90 * deg)[0] + joint(rho_pair(), a, 90 * deg)[2], 0.5) for a in np.linspace(0, np.pi, 7))
print('All checks passed.')

## Things to try

Each idea matches a "Predict first" question in the [browser lab](https://tonylowgroup.github.io/Professor-Nano-Labs/labs/entanglement-bell-test/).

1. **Other Bell states.** Replace the singlet with $(|{\uparrow}{\uparrow}\rangle+|{\downarrow}{\downarrow}\rangle)/\sqrt2$. How does $E(a,b)$ change, and which dials maximize the CHSH score now?
2. **A better answer sheet?** Change the hidden-arrow rule (for example, let Bob answer $-\operatorname{sign}\cos(b-\lambda+0.3)$). Can any rule beat 2? Check numerically.
3. **Noise in the dials.** Add a small random error to each dial angle in `play`. How fast does the score fall?
4. **Steering in 3D.** Let Alice measure along y. Where do Bob's two conditional Bloch vectors point, and what is their average?